In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime


In [ ]:

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_data_warehouse"
CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.Customer_Staging"
wh_table_name = "reporting.dimension.Customers"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog("local")

spark

# <center> Data Flow Diagram </center>
```mermaid
flowchart TB

p_BusinessEntityID -- INSERT ---> cust_stg_customer_id
p_Title -- INSERT ---> cust_stg_title
p_FirstName -- INSERT ---> cust_stg_firstname
p_MiddleName -- INSERT ---> cust_stg_middlename
p_LastName -- INSERT ---> cust_stg_lastname
p_Suffix -- INSERT ---> cust_stg_suffix
pp_PhoneNumber -- INSERT ---> cust_stg_phonenumber
pnt_Name -- INSERT ---> cust_stg_phonenumbertype
ea_EmailAddress -- INSERT ---> cust_stg_emailaddress
p_EmailPromotion -- INSERT ---> cust_stg_emailpromotion
at_Name -- INSERT ---> cust_stg_addresstype
a_AddressLine1 -- INSERT ---> cust_stg_addressline1
a_AddressLine2 -- INSERT ---> cust_stg_addressline2
a_City -- INSERT ---> cust_stg_city
sp_Name -- INSERT ---> cust_stg_StateProvinceName
a_PostalCode -- INSERT ---> cust_stg_postalcode
cr_Name -- INSERT ---> cust_stg_countryregionname
p_demographics -- INSERT ---> cust_stg_demographics

cust_stg_customer_key ins1@-- UPSERT --> cust_dim_customer_key
cust_stg_customer_id ins2@-- UPSERT --> cust_dim_customer_id
cust_stg_title ins3@-- UPSERT --> cust_dim_title
cust_stg_firstname ins4@-- UPSERT --> cust_dim_firstname
cust_stg_middlename ins5@-- UPSERT --> cust_dim_middlename
cust_stg_lastname ins6@-- UPSERT --> cust_dim_lastname
cust_stg_suffix ins7@-- UPSERT --> cust_dim_suffix
cust_stg_phonenumber ins8@-- UPSERT --> cust_dim_phonenumber
cust_stg_phonenumbertype ins9@-- UPSERT --> cust_dim_phonenumbertype
cust_stg_emailaddress ins10@-- UPSERT --> cust_dim_emailaddress
cust_stg_emailpromotion ins11@-- UPSERT --> cust_dim_emailpromotion
cust_stg_addresstype ins12@-- UPSERT --> cust_dim_addresstype
cust_stg_addressline1 ins13@-- UPSERT --> cust_dim_addressline1
cust_stg_addressline2 ins14@-- UPSERT --> cust_dim_addressline2
cust_stg_city ins15@-- UPSERT --> cust_dim_city
cust_stg_StateProvinceName ins16@-- UPSERT --> cust_dim_StateProvinceName
cust_stg_postalcode ins17@-- UPSERT --> cust_dim_postalcode
cust_stg_countryregionname ins18@-- UPSERT --> cust_dim_countryregionname
cust_stg_demographics ins19@-- UPSERT --> cust_dim_demographics

ins1@{animation: fast}
ins2@{animation: fast}
ins3@{animation: fast}
ins4@{animation: fast}
ins5@{animation: fast}
ins6@{animation: fast}
ins7@{animation: fast}
ins8@{animation: fast}
ins9@{animation: fast}
ins10@{animation: fast}
ins11@{animation: fast}
ins12@{animation: fast}
ins13@{animation: fast}
ins14@{animation: fast}
ins15@{animation: fast}
ins16@{animation: fast}
ins17@{animation: fast}
ins18@{animation: fast}
ins19@{animation: fast}


p_BusinessEntityID j1@o-.JOIN.-o bea_BusinessEntityID
j1@{animation: slow}
a_AddressID j2@o-.JOIN.-o bea_AddressID
j2@{animation: slow}
sp_StateProvinceID j3@o-.JOIN.-o a_StateProvinceID
j3@{animation: slow}
c_PersonID j4@o-.JOIN.-o p_BusinessEntityID
j4@{animation: slow}
ea_BusinessEntityID j5@o-.JOIN.-o p_BusinessEntityID
j5@{animation: slow}
pp_BusinessEntityID j6@o-.JOIN.-o p_BusinessEntityID
j6@{animation: slow}
pp_PhoneNumberTypeID j7@o-.JOIN.-o pnt_PhoneNumberTypeID
j7@{animation: slow}
at_AddressTypeID j8@o-.JOIN.-o bea_AddressTypeID
j8@{animation: slow}
cr_CountryRegionCode j9@o-.JOIN.-o sp_CountryRegionCode
j9@{animation: slow}

    subgraph Destination
        subgraph customer staging
            direction LR
            cust_stg_customer_key[customer_key]
            cust_stg_customer_id[customer_id]
            cust_stg_title[title]
            cust_stg_firstname[firstname]
            cust_stg_middlename[middlename]
            cust_stg_lastname[lastname]
            cust_stg_suffix[suffix]
            cust_stg_phonenumber[phonenumber]
            cust_stg_phonenumbertype[phonenumbertype]
            cust_stg_emailaddress[emailaddress]
            cust_stg_emailpromotion[emailpromotion]
            cust_stg_addresstype[addresstype]
            cust_stg_addressline1[addressline1]
            cust_stg_addressline2[addressline2]
            cust_stg_city[city]
            cust_stg_StateProvinceName[StateProvinceName]
            cust_stg_postalcode[postalcode]
            cust_stg_countryregionname[countryregionname]
            cust_stg_demographics[demographics]
        end
        subgraph Dimension.customer
            direction LR
            cust_dim_customer_key[customer_key]
            cust_dim_customer_id[customer_id]
            cust_dim_title[title]
            cust_dim_firstname[firstname]
            cust_dim_middlename[middlename]
            cust_dim_lastname[lastname]
            cust_dim_suffix[suffix]
            cust_dim_phonenumber[phonenumber]
            cust_dim_phonenumbertype[phonenumbertype]
            cust_dim_emailaddress[emailaddress]
            cust_dim_emailpromotion[emailpromotion]
            cust_dim_addresstype[addresstype]
            cust_dim_addressline1[addressline1]
            cust_dim_addressline2[addressline2]
            cust_dim_city[city]
            cust_dim_StateProvinceName[StateProvinceName]
            cust_dim_postalcode[postalcode]
            cust_dim_countryregionname[countryregionname]
            cust_dim_demographics[demographics]
        end
    end
    subgraph Source
        subgraph    Person.Person
        direction LR
            p_BusinessEntityID[BusinessEntityID] 
            p_Title[Title] 
            p_FirstName[FirstName] 
            p_MiddleName[MiddleName] 
            p_LastName[LastName] 
            p_Suffix[Suffix]
            p_EmailPromotion[EmailPromotion] 
            p_AdditionalContactInfo[AdditionalContactInfo] 
            p_demographics[Demographics] 
        end
        subgraph    Person.BusinessEntityAddress
        direction LR
            bea_BusinessEntityID[BusinessEntityID]
            bea_AddressID[AddressID]
            bea_AddressTypeID[AddressTypeID]
        end
        subgraph    Person.Address
        direction LR
            a_AddressID[AddressID]
            a_AddressLine1[AddressLine1]
            a_AddressLine2[AddressLine2]
            a_City[City]
            a_StateProvinceID[StateProvinceID]
            a_PostalCode[PostalCode]
        end
        subgraph Person.AddressType
            direction LR
            at_Name[Name] 
            at_AddressTypeID[AddressTypeID]
        end
        subgraph Sales.Customer
            direction LR
            c_PersonID[PersonID]
            c_StoreID[StoreID]
        end
        subgraph Person.EmailAddress
            direction LR
            ea_BusinessEntityID[BusinessEntityID]
            ea_EmailAddress[EmailAddress]
        end
        subgraph Person.PersonPhone
            direction LR
            pp_PhoneNumber[PhoneNumber]
            pp_BusinessEntityID[BusinessEntityID]
            pp_PhoneNumberTypeID[PhoneNumberTypeID]
        end
        subgraph Person.PhoneNumberType
            direction LR
            pnt_Name[Name] 
            pnt_PhoneNumberTypeID[PhoneNumberTypeID]
        end
        subgraph Person.StateProvince
            direction LR
            sp_Name[Name]
            sp_StateProvinceCode[Subregion] 
            sp_StateProvinceID[StateProvinceID] 
            sp_TerritoryID[TerritoryID] 
            sp_CountryRegionCode[CountryRegionCode] 
        end
        subgraph Person.CountryRegion
            direction LR
            cr_Name[Name]
            cr_CountryRegionCode[CountryRegionCode] 
            
        end
    end
```

In [ ]:
# Load and alias all required tables
p     = spark.table("Person.Person").alias("p")
bea   = spark.table("Person.BusinessEntityAddress").alias("bea")
a     = spark.table("Person.Address").alias("a")
sp    = spark.table("Person.StateProvince").alias("sp")
cr    = spark.table("Person.CountryRegion").alias("cr")
at    = spark.table("Person.AddressType").alias("at")
c     = spark.table("Sales.Customer").alias("c")
ea    = spark.table("Person.EmailAddress").alias("ea")
pp    = spark.table("Person.PersonPhone").alias("pp")
pnt   = spark.table("Person.PhoneNumberType").alias("pnt")

# customer = customer.withColumn(
#         "StoreID",
#         sf.when(sf.lower(sf.col("StoreID")) == "null", None).otherwise(sf.col("StoreID"))
#     )

# Build the join chain
joined = (
    p.join(bea, p["BusinessEntityID"] == bea["BusinessEntityID"], "inner")
     .join(a, bea["AddressID"] == a["AddressID"], "inner")
     .join(sp, a["StateProvinceID"] == sp["StateProvinceID"], "inner")
     .join(cr, sp["CountryRegionCode"] == cr["CountryRegionCode"], "inner")
     .join(at, bea["AddressTypeID"] == at["AddressTypeID"], "inner")
     .join(c, c["PersonID"] == p["BusinessEntityID"], "inner")
     .join(ea, ea["BusinessEntityID"] == p["BusinessEntityID"], "left_outer")
     .join(pp, pp["BusinessEntityID"] == p["BusinessEntityID"], "left_outer")
     .join(pnt, pnt["PhoneNumberTypeID"] == pp["PhoneNumberTypeID"], "left_outer")
    .filter((sf.col("StoreID").isNull()) | (sf.lower(sf.col("StoreID")) == "null"))
)

joined =joined\
    .withColumn("Customer_Key", (sf.monotonically_increasing_id() + 1))

# Select the required columns
customer_data = joined.select(
    sf.col("Customer_Key").cast("int").alias("customer_key"),
    p["BusinessEntityID"].cast("int").alias("customer_id"),
    p["Title"].alias("title"),
    p["FirstName"].alias("firstname"),
    p["MiddleName"].alias("middlename"),
    p["LastName"].alias("lastname"),
    p["Suffix"].alias("suffix"),
    pp["PhoneNumber"].alias("phonenumber"),
    pnt["Name"].alias("phonenumbertype"),
    ea["EmailAddress"].alias("emailaddress"),
    p["EmailPromotion"].alias("emailpromotion"),
    at["Name"].alias("addresstype"),
    a["AddressLine1"].alias("addressline1"),
    a["AddressLine2"].alias("addressline2"),
    a["City"].alias("city"),
    sp["Name"].alias("StateProvinceName"),
    a["PostalCode"].alias("postalcode"),
    cr["Name"].alias("countryregionname"),
    p["Demographics"].alias("demographics"),
)



# Show sample output
# customer_data.show(10, truncate=False)
# customer_data.printSchema()

customer_data.writeTo(staging_table_name) \
    .partitionedBy("countryregionname") \
    .using("iceberg") \
    .createOrReplace()

# customer_data.count()

In [ ]:
# # spark.sql("SHOW TBLPROPERTIES " + staging_table_name).show(truncate=False)
# # spark.sql("DESCRIBE TABLE EXTENDED " + staging_table_name).show(30, truncate=False)
# spark.read.table(staging_table_name + ".partitions").show(truncate=False)

In [ ]:
# spark.sql("Select * from " + staging_table_name + " LIMIT 20").show(truncate=False)


In [ ]:
stage_df = spark.table(staging_table_name)
stage_df.printSchema()
stage_df = stage_df\
    .withColumn("record_hash", 
                sf.sha2(sf.concat(
                    sf.coalesce(sf.col("title"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("firstname"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("middlename"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("lastname"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("suffix"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("phonenumber"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("phonenumbertype"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("emailaddress"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("emailpromotion"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addresstype"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addressline1"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addressline2"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("city"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("StateProvinceName"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("postalcode"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("countryregionname"),sf.lit("")), sf.lit("|"),
                    ),
                    256)
                )

stage_df.select(
    "customer_key",
     "customer_id",
     "title",
     "firstname",
     "middlename",
     "lastname",
     "record_hash",     
).show(10, truncate=False)




In [ ]:
df_destination = spark.range(0)
df_destination = df_destination\
    .withColumn("customer_key", sf.lit(None).cast(sdt.IntegerType()))\
    .withColumn("customer_id", sf.lit(None).cast(sdt.IntegerType()))\
    .withColumn("title", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("firstname", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("middlename", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("lastname", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("suffix", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("phonenumber", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("phonenumbertype", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("emailaddress", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("emailpromotion", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("addresstype", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("addressline1", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("addressline2", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("city", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("StateProvinceName", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("postalcode", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("countryregionname", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("demographics", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("record_hash", sf.lit(None).cast(sdt.StringType()))\
    .withColumn("validFrom", sf.lit(None).cast(sdt.DateType()))\
    .withColumn("validTill", sf.lit(None).cast(sdt.DateType()))\
    .withColumn("isActive", sf.lit(None).cast(sdt.BooleanType()))

df_destination = df_destination.drop("id")

df_destination.printSchema()

df_destination.writeTo(wh_table_name) \
    .partitionedBy("countryregionname") \
    .using("iceberg") \
    .createOrReplace()


In [ ]:
df_source = spark.table(staging_table_name)
df_source = df_source\
    .withColumn("record_hash", 
                sf.sha2(sf.concat(
                    sf.coalesce(sf.col("title"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("firstname"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("middlename"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("lastname"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("suffix"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("phonenumber"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("phonenumbertype"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("emailaddress"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("emailpromotion"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addresstype"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addressline1"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addressline2"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("city"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("StateProvinceName"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("postalcode"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("countryregionname"),sf.lit("")), sf.lit("|"),
                    ),
                    256)
                ).alias("scrc")

df_destination = spark.table(wh_table_name).alias("dest")

df_source.show(10, truncate=False)

df_insert = df_source\
    .join(df_destination,
          (
              (sf.col("scrc.customer_id") == sf.col("dest.customer_id")) & 
              (sf.col("scrc.record_hash") == sf.col("dest.record_hash"))
        ),
          how = "left"
          )\
    .filter(sf.col("dest.record_hash").isNull()) \
    .select("scrc.*")
    
df_insert = df_insert\
    .withColumn("validFrom", sf.date_add(sf.lit(datetime.now().date()), -2))\
    .withColumn("validTill", sf.lit(None).cast(sdt.DateType()))\
    .withColumn("isActive", sf.lit(True))


df_insert = df_insert.alias("new_update")

df_Update = df_destination\
    .join(df_insert,
          sf.col("dest.customer_id") == sf.col("new_update.customer_id"),
          how = "inner"
          )\
    .select(
        "dest.*", 
        )

df_insert.show(10, truncate=False)

df_Update = df_Update\
    .withColumn("validTill", sf.date_add(sf.col("validFrom"), 1))\
    .withColumn("isActive", sf.lit(False))

df_Update.show(10, truncate=False)

In [ ]:
df_Update.write \
    .format("iceberg") \
    .mode("overwrite") \
    .save(wh_table_name)

df_insert.write \
    .format("iceberg") \
    .mode("append") \
    .save(wh_table_name)

In [ ]:
spark.table(wh_table_name).count()

# spark.sql("SHOW TBLPROPERTIES " + wh_table_name).show(truncate=False)
# spark.sql("DESCRIBE TABLE EXTENDED " + wh_table_name).show(truncate=False)
# spark.read.table(wh_table_name + ".partitions").show(truncate=False)

In [ ]:
spark.sql(
    "UPDATE " + staging_table_name +
    " SET "
    "title = NULLIF(title, 'NULL'),"
    "firstname = NULLIF(firstname, 'NULL'),"
    "middlename = NULLIF(middlename, 'NULL'),"
    "lastname = NULLIF(lastname, 'NULL'),"
    "suffix = NULLIF(suffix, 'NULL'),"
    "phonenumber = NULLIF(phonenumber, 'NULL'),"
    "phonenumbertype = NULLIF(phonenumbertype, 'NULL'),"
    "emailaddress = NULLIF(emailaddress, 'NULL'),"
    "emailpromotion = NULLIF(emailpromotion, 'NULL'),"
    "addresstype = NULLIF(addresstype, 'NULL'),"
    "addressline1 = NULLIF(addressline1, 'NULL'),"
    "addressline2 = NULLIF(addressline2, 'NULL'),"
    "city = NULLIF(city, 'NULL'),"
    "StateProvinceName = NULLIF(StateProvinceName, 'NULL'),"
    "postalcode = NULLIF(postalcode, 'NULL'),"
    "countryregionname = NULLIF(countryregionname, 'NULL')"
    )
spark.sql("Select * from " + staging_table_name + " LIMIT 20").show(truncate=False)

In [ ]:
df_source = spark.table(staging_table_name).alias("scrc")
df_source = df_source\
    .withColumn("record_hash", 
                sf.sha2(sf.concat(
                    sf.coalesce(sf.col("title"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("firstname"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("middlename"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("lastname"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("suffix"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("phonenumber"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("phonenumbertype"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("emailaddress"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("emailpromotion"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addresstype"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addressline1"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("addressline2"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("city"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("StateProvinceName"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("postalcode"),sf.lit("")), sf.lit("|"),
                    sf.coalesce(sf.col("countryregionname"),sf.lit("")), sf.lit("|"),
                    ),
                    256)
                ).alias("scrc")

df_destination = spark.table(wh_table_name).alias("dest")

# df_source.show(10, truncate=False)

df_insert = df_source\
    .join(df_destination,
          (
              (sf.col("scrc.customer_id") == sf.col("dest.customer_id")) & 
              (sf.col("scrc.record_hash") == sf.col("dest.record_hash")) & 
              (sf.col("dest.isActive") == sf.lit("True"))
        ),
          how = "left"
          )\
    .filter(sf.col("dest.record_hash").isNull()) \
    .select("scrc.*")
    
df_insert = df_insert\
    .withColumn("validFrom", sf.date_add(sf.lit(datetime.now().date()), -1))\
    .withColumn("validTill", sf.lit(None).cast(sdt.DateType()))\
    .withColumn("isActive", sf.lit(True))


df_insert = df_insert.alias("new_update")

df_Update = df_destination\
    .join(df_insert,
          sf.col("dest.customer_id") == sf.col("new_update.customer_id"),
          how = "inner"
          )\
    .select(
        "dest.*", 
        )

df_insert.show(10, truncate=False)

df_Update = df_Update\
    .withColumn("validTill", sf.date_add(sf.col("validFrom"), 1))\
    .withColumn("isActive", sf.lit(False))

df_Update.show(10, truncate=False)

df_Update.write \
    .format("iceberg") \
    .mode("overwrite") \
    .save(wh_table_name)

df_insert.write \
    .format("iceberg") \
    .mode("append") \
    .save(wh_table_name)

In [ ]:
# Assuming you have a SparkSession object named 'spark'

# spark.sql("DELETE FROM "+ wh_table_name +" WHERE 1=1 AND customer_key > 550")

spark.sql("Select * FROM "+ wh_table_name +" WHERE 1=1 AND customer_key = 587").show()

# spark.sql("DELETE FROM "+ wh_table_name +" WHERE 1 = 1")


In [ ]:
spark.stop()

In [ ]:
dfinf = spark.table(wh_table_name)
# print(dfinf._jdf.queryExecution().logical())
# print(dfinf._jdf.queryExecution().analyzed())
# print(dfinf._jdf.queryExecution().executedPlan())
# dfinf.explain(True)

# dfinf is your DataFrame
logical = dfinf._jdf.queryExecution().logical()

# Cast to UnresolvedRelation and get the multipart identifier
multipart_ident = logical.tableIdentifier().quotedString()

print(multipart_ident)  # -> reporting.dimension.Customers


